In [ ]:
# ✅ Instalar dependências
!pip install diffusers==0.33.1 transformers accelerate einops gradio ffmpeg-python safetensors --quiet

In [ ]:
# 🔐 Login na Hugging Face
from huggingface_hub import login
login()

In [ ]:
# 📦 Imports e configurações gerais
import os
import torch
from PIL import Image
import gradio as gr
from diffusers import CogVideoXImageToVideoPipeline
from diffusers.utils import export_to_video, load_image
import glob
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs("outputs", exist_ok=True)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

In [ ]:
# ✅ Carregar componentes com controle de memória
from transformers import logging
logging.set_verbosity_error()
from diffusers import CogVideoXTransformer3DModel, AutoencoderKLCogVideoX
transformer = CogVideoXTransformer3DModel.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    subfolder="transformer",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="balanced",
    offload_folder="modelo_offload"
)
vae = AutoencoderKLCogVideoX.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    subfolder="vae",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="balanced",
    offload_folder="modelo_offload"
)

In [ ]:
# ✅ Carregar tokenizer, text_encoder e scheduler
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import DDIMScheduler
text_encoder = CLIPTextModel.from_pretrained(
    "openai/clip-vit-large-patch14",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="balanced",
    offload_folder="modelo_offload"
)
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
scheduler = DDIMScheduler.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    subfolder="scheduler"
)

In [ ]:
# 🧠 Montar o pipeline completo
pipe = CogVideoXImageToVideoPipeline(
    transformer=transformer,
    vae=vae,
    tokenizer=tokenizer,
    text_encoder=text_encoder,
    scheduler=scheduler
).to(device)

In [ ]:
# 🎥 Função de geração com tratamento de erros
def generate_video(image_path, prompt, width, height, frames, steps):
    try:
        if not os.path.exists(image_path):
            raise ValueError("Imagem não encontrada no caminho fornecido.")
        for f in glob.glob("outputs/*.mp4"):
            os.remove(f)
        image = load_image(image_path).convert("RGB").resize((width, height))
        result = pipe(
            image=image,
            prompt=prompt,
            guidance_scale=5,
            num_inference_steps=steps,
            num_frames=frames
        )
        frames_result = result.frames[0]
        out_path = f"outputs/cogvideo_{width}x{height}.mp4"
        export_to_video(frames_result, out_path, fps=6)
        return out_path
    except Exception as e:
        print(f"Erro durante a geração: {e}")
        return None

In [ ]:
# 🖼️ Interface Gradio com status
with gr.Blocks() as demo:
    gr.Markdown("## CogVideoX - Geração de vídeo a partir de imagem 🎬")
    with gr.Row():
        img = gr.Image(type="filepath", label="Imagem")
        prm = gr.Textbox(label="Prompt (ex: 'neve caindo à noite')")
    with gr.Row():
        width = gr.Slider(256, 1280, value=720, step=64, label="Largura")
        height = gr.Slider(256, 720, value=480, step=64, label="Altura")
    with gr.Row():
        frames = gr.Slider(2, 16, value=8, step=1, label="Nº de Frames")
        steps = gr.Slider(4, 50, value=16, step=1, label="Inference Steps")
    btn = gr.Button("🎬 Gerar Vídeo")
    vid = gr.Video(label="🎞️ Resultado")
    status = gr.Markdown("Aguardando...")
    def wrapper_generate(image_path, prompt, width, height, frames, steps):
        status.update("Gerando vídeo... Aguarde...")
        video_path = generate_video(image_path, prompt, width, height, frames, steps)
        if video_path:
            status.update("Vídeo gerado com sucesso!")
            return video_path
        else:
            status.update("Erro ao gerar o vídeo. Verifique o console.")
            return None
    btn.click(fn=wrapper_generate, inputs=[img, prm, width, height, frames, steps], outputs=vid)
    demo.launch(share=True, debug=False)